# EfficientNetV2 Training Script on Cifar

This is basic tutorial on how to training EfficientNetV2 on Cifar10/100.

We use Pytorch-Lightning. If you are unfamiliar with pytorch-lightning, please read [Lightning-in-2-steps](https://pytorch-lightning.readthedocs.io/en/latest/starter/new-project.html).




### 1. Install required packages

In [1]:
!pip3 install pytorch_lightning

### 2. Configuration

In [2]:
config = {'seed': 2021,
          'trainer': {
              'max_epochs': 15,
              'devices': 1,  # Specify the number of devices (e.g., 1 for a single GPU)
              'accelerator': 'gpu',  # Use 'gpu' for GPU training or 'cpu' for CPU
              'accumulate_grad_batches': 1,
              'fast_dev_run': False,
              'num_sanity_val_steps': 0,
          },
          'data': {
              'dataset_name': 'cifar100',
              'batch_size': 128,
              'num_workers': 4,
              'size': [224, 224],
              'data_root': 'data',
              'valid_ratio': 0.1
          },
          'model':{
                'backbone_init': {
                    'model': 'efficientnet_v2_s_in21k',
                    'nclass': 0, # do not change this
                    'pretrained': True,
                    },
                'optimizer_init':{
                    'class_path': 'torch.optim.SGD',
                    'init_args': {
                        'lr': 0.001,
                        'momentum': 0.95,
                        'weight_decay': 0.0005
                        }
                    },
                'lr_scheduler_init':{
                    'class_path': 'torch.optim.lr_scheduler.CosineAnnealingLR',
                    'init_args':{
                        'T_max': 0 # no need to change this
                        }
                    }
            }
}

### 3. Load DataModule

In [3]:
from typing import Type, Any

from pytorch_lightning import LightningDataModule
from pytorch_lightning.utilities.types import TRAIN_DATALOADERS, EVAL_DATALOADERS

from torchvision import transforms
from torchvision.datasets import CIFAR10, CIFAR100

from torch.utils.data import random_split, DataLoader
import numpy as np

class BaseDataModule(LightningDataModule):
    def __init__(self,
                 dataset_name: str,
                 dataset: Type[Any],
                 train_transform: Type[Any],
                 test_transform: Type[Any],
                 batch_size: int = 64,
                 num_workers: int = 4,
                 data_root: str = 'data',
                 valid_ratio: float = 0.1):
        """
        Base Data Module
        :arg
            Dataset: Enter Dataset
            batch_size: Enter batch size
            num_workers: Enter number of workers
            size: Enter resized image
            data_root: Enter root data folder name
            valid_ratio: Enter valid dataset ratio
        """
        super(BaseDataModule, self).__init__()
        self.dataset_name = dataset_name
        self.dataset = dataset
        self.train_transform = train_transform
        self.test_transform = test_transform
        self.batch_size = batch_size
        self.num_workers = num_workers
        self.data_root = data_root
        self.valid_ratio = valid_ratio
        self.num_classes = None
        self.num_step = None
        self.prepare_data()

    def prepare_data(self) -> None:
        train = self.dataset(root=self.data_root, train=True, download=True)
        test = self.dataset(root=self.data_root, train=False, download=True)
        self.num_classes = len(train.classes)
        self.num_step = len(train) // self.batch_size

        print('-' * 50)
        print('* {} dataset class num: {}'.format(self.dataset_name, len(train.classes)))
        print('* {} train dataset len: {}'.format(self.dataset_name, len(train)))
        print('* {} test dataset len: {}'.format(self.dataset_name, len(test)))
        print('-' * 50)

    def setup(self, stage: str = None):
        if stage in (None, 'fit'):
            ds = self.dataset(root=self.data_root, train=True, transform=self.train_transform)
            self.train_ds, self.valid_ds = self.split_train_valid(ds)

        elif stage in (None, 'test', 'predict'):
            self.test_ds = self.dataset(root=self.data_root, train=False, transform=self.test_transform)

    def split_train_valid(self, ds):
        ds_len = len(ds)
        valid_ds_len = int(ds_len * self.valid_ratio)
        train_ds_len = ds_len - valid_ds_len
        return random_split(ds, [train_ds_len, valid_ds_len])

    def train_dataloader(self) -> TRAIN_DATALOADERS:
        return DataLoader(self.train_ds, batch_size=self.batch_size, shuffle=True, num_workers=self.num_workers)

    def val_dataloader(self) -> EVAL_DATALOADERS:
        return DataLoader(self.valid_ds, batch_size=self.batch_size, shuffle=False, num_workers=self.num_workers)

    def test_dataloader(self) -> EVAL_DATALOADERS:
        return DataLoader(self.test_ds, batch_size=self.batch_size, shuffle=False, num_workers=self.num_workers)

    def predict_dataloader(self) -> EVAL_DATALOADERS:
        return DataLoader(self.test_ds, batch_size=self.batch_size, shuffle=False, num_workers=self.num_workers)


class CIFAR(BaseDataModule):
    def __init__(self, dataset_name: str, size: tuple, **kwargs):
        if dataset_name == 'cifar10':
            dataset, mean, std = CIFAR10, (0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616)
        elif dataset_name == 'cifar100':
            dataset, mean, std = CIFAR100, (0.5071, 0.4865, 0.4409), (0.2673, 0.2564, 0.2762)

        train_transform, test_transform = self.get_trasnforms(mean, std, size)
        super(CIFAR, self).__init__(dataset_name, dataset, train_transform, test_transform, **kwargs)

    def get_trasnforms(self, mean, std, size):
        train = transforms.Compose([
            transforms.Resize(size),
            transforms.Pad(4, padding_mode='reflect'),
            transforms.RandomCrop(size),
            transforms.RandomHorizontalFlip(),
            transforms.ToTensor(),
            transforms.Normalize(mean=mean, std=std)
        ])
        test = transforms.Compose([
            transforms.Resize(size),
            transforms.CenterCrop(size),
            transforms.ToTensor(),
            transforms.Normalize(mean=mean, std=std)
        ])
        return train, test

### 4. Train & Val & Test Model

In [6]:
import os
import math
import warnings

import torch
from torch import nn
from torch.optim import SGD

from pytorch_lightning import LightningModule, Trainer
from pytorch_lightning.cli import instantiate_class, LightningCLI
from torchmetrics import MetricCollection, Accuracy
from pytorch_lightning.callbacks import Callback
import numpy as np

class AccuracyLoggerCallback(Callback):
    def __init__(self):
        self.train_accuracies = []
        self.val_accuracies = []

    def on_train_epoch_end(self, trainer, pl_module):
        print("Training Log")
        # Get training accuracy from the logs
        train_acc = trainer.logged_metrics.get('train/top1')
        
        if train_acc is not None:
            self.train_accuracies.append(train_acc.item())
        else:
            print("Something went wrong in Train Logging")
            
    def on_validation_epoch_end(self, trainer, pl_module):
        print("Validation Log")

        val_acc = trainer.logged_metrics.get('valid/top1')

        if val_acc is not None:
            self.val_accuracies.append(val_acc.item())
        else:
            print("Something went wrong in Val Logging")

    def get_train_accuracies(self):
        return self.train_accuracies

    def get_val_accuracies(self):
        return self.val_accuracies



# Initialize global counters
total_examples_seen = 0
total_examples_used = 0

def adaptive_sorting_loss(output, target):
    """
    Custom loss function that selects either the top half of the training examples
    or the top 2/3 of the cumulative loss sum, depending on which is larger.
    Updates global counters to track total examples and examples used.
    """
    global total_examples_seen, total_examples_used

    # Compute per-sample cross-entropy losses
    ce_loss = nn.CrossEntropyLoss(reduction='none')(output, target)

    # Total loss sum
    Z = ce_loss.sum()

    # Sort losses in descending order
    sorted_losses, _ = torch.sort(ce_loss, descending=True)

    # Compute cumulative sum
    cumulative_sum = torch.cumsum(sorted_losses, dim=0)

    # Determine the threshold for top 2/3 of the loss sum
    threshold = (2 / 3) * Z

    # Mask to select losses within the threshold
    mask = cumulative_sum <= threshold

    # Count the number of elements in the mask
    mask_count = mask.sum().item()

    # Update the global counter for total examples seen
    total_examples_seen += len(sorted_losses)

    # Update the global counter for examples used
    if mask_count < len(sorted_losses) // 2:
        # Take the mean of the top half of the losses
        final_loss = sorted_losses[:len(sorted_losses) // 2].mean()
        total_examples_used += len(sorted_losses) // 2
    else:
        # Otherwise, return the mean of the masked losses
        selected_losses = sorted_losses[mask]
        final_loss = selected_losses.mean()
        total_examples_used += mask_count

    return final_loss

class BaseVisionSystem(LightningModule):
    def __init__(self, backbone_init: dict, num_classes: int, num_step: int, accelerator: str, devices: int, max_epochs: int,
                 optimizer_init: dict, lr_scheduler_init: dict):
        super(BaseVisionSystem, self).__init__()

        # Initialize the rest of your model
        self.num_step = num_step
        self.max_epochs = max_epochs
        self.backbone = torch.hub.load('hankyul2/EfficientNetV2-pytorch', **backbone_init)
        self.fc = nn.Linear(self.backbone.out_channels, num_classes)

        # Other initializations...
        self.optimizer_init_config = optimizer_init
        self.lr_scheduler_init_config = lr_scheduler_init
        self.criterion = nn.CrossEntropyLoss()

        # Save accelerator and devices
        self.accelerator = accelerator
        self.devices = devices

        # Set automatic optimization to False
        self.automatic_optimization = False

        # Define metrics
        metrics = MetricCollection({
            'top1': Accuracy(task='multiclass', top_k=1, num_classes=num_classes),
            'top5': Accuracy(task='multiclass', top_k=5, num_classes=num_classes),
        })
        self.train_metric = metrics.clone(prefix='train/')
        self.valid_metric = metrics.clone(prefix='valid/')


    def forward(self, x):
        return self.fc(self.backbone(x))

    def training_step(self, batch, batch_idx):
        # Manually handle optimization
        x, y = batch
        loss, y_hat = self.compute_loss(x, y)

        # Manually optimize each optimizer
        optimizer = self.optimizers()
        optimizer.zero_grad()
        self.manual_backward(loss)
        optimizer.step()

        # Log metrics
        self.train_metric(y_hat, y)
        self.log_dict({'train/loss': loss, **self.train_metric}, prog_bar=True)
        return loss

    def validation_step(self, batch, batch_idx):
        return self.shared_step(batch, self.valid_metric, 'valid', add_dataloader_idx=True)

    def test_step(self, batch, batch_idx):
        return self.shared_step(batch, self.test_metric, 'test', add_dataloader_idx=True)

    def shared_step(self, batch, metric, mode, add_dataloader_idx):
        x, y = batch
        loss, y_hat = self.compute_loss_eval(x, y)
        metric(y_hat, y)
        self.log_dict({f'{mode}/loss': loss}, add_dataloader_idx=add_dataloader_idx)
        self.log_dict(metric, add_dataloader_idx=add_dataloader_idx, prog_bar=True)
        return loss

    def compute_loss(self, x, y):
        return self.compute_loss_eval(x, y)

    def compute_loss_eval(self, x, y):
        y_hat = self.fc(self.backbone(x))
        loss = self.criterion(y_hat, y)
        return loss, y_hat

    def configure_optimizers(self):
        optimizer = instantiate_class([
            {'params': self.backbone.parameters(), 'lr': self.optimizer_init_config['init_args']['lr'] * 0.1},
            {'params': self.fc.parameters()},
        ], self.optimizer_init_config)

        lr_scheduler = {
            'scheduler': instantiate_class(optimizer, self.update_and_get_lr_scheduler_config()),
            'interval': 'step'
        }
        return {'optimizer': optimizer, 'lr_scheduler': lr_scheduler}

    def update_and_get_lr_scheduler_config(self):
        if 'T_max' in self.lr_scheduler_init_config['init_args']:
            self.lr_scheduler_init_config['init_args']['T_max'] = self.num_step * self.max_epochs
        return self.lr_scheduler_init_config

def update_config(config, data):
    config['model']['num_classes'] = data.num_classes
    config['model']['num_step'] = data.num_step
    config['model']['max_epochs'] = config['trainer']['max_epochs']
    config['model']['accelerator'] = config['trainer']['accelerator']
    config['model']['devices'] = config['trainer']['devices']




if __name__ == '__main__':
    data = CIFAR(**config['data'])
    update_config(config, data)
    model = BaseVisionSystem(**config['model'])
    
    # Create an instance of the callback
    accuracy_logger = AccuracyLoggerCallback()

    trainer = Trainer(**config['trainer'],callbacks = [accuracy_logger])
    trainer.fit(model, data)


Files already downloaded and verified
Files already downloaded and verified
--------------------------------------------------
* cifar100 dataset class num: 100
* cifar100 train dataset len: 50000
* cifar100 test dataset len: 10000
--------------------------------------------------


Using cache found in /home/user/.cache/torch/hub/hankyul2_EfficientNetV2-pytorch_main
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs


Files already downloaded and verified
Files already downloaded and verified
--------------------------------------------------
* cifar100 dataset class num: 100
* cifar100 train dataset len: 50000
* cifar100 test dataset len: 10000
--------------------------------------------------


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name         | Type             | Params | Mode 
----------------------------------------------------------
0 | backbone     | EfficientNetV2   | 20.2 M | train
1 | fc           | Linear           | 128 K  | train
2 | criterion    | CrossEntropyLoss | 0      | train
3 | train_metric | MetricCollection | 0      | train
4 | valid_metric | MetricCollection | 0      | train
----------------------------------------------------------
20.3 M    Trainable params
0         Non-trainable params
20.3 M    Total params
81.222    Total estimated model params size (MB)
755       Modules in train mode
0         Modules in eval mode


Training: |                                               | 0/? [00:00<?, ?it/s]

...

Validation: |                                             | 0/? [00:00<?, ?it/s]

Validation Log
Training Log


Validation: |                                             | 0/? [00:00<?, ?it/s]

Validation Log
Training Log


`Trainer.fit` stopped: `max_epochs=15` reached.


In [23]:
train_acc = accuracy_logger.get_train_accuracies()
val_acc = accuracy_logger.get_val_accuracies()
print(train_acc)
# print(val_acc)
val_acc = [acc.cpu().numpy() for acc in val_acc]
print(val_acc)
# # Save the accuracies as .npy files
# np.save('vanilla_train_acc.npy', train_acc)
# np.save('vanilla_val_acc.npy', val_acc)


[0.4027777910232544, 0.75, 0.8055555820465088, 0.7777777910232544, 0.7916666865348816, 0.8611111044883728, 0.8194444179534912, 0.8472222089767456, 0.7361111044883728, 0.8472222089767456, 0.8472222089767456, 0.8472222089767456, 0.8055555820465088, 0.7638888955116272, 0.8611111044883728]
[array(0.6358, dtype=float32), array(0.7638, dtype=float32), array(0.7992, dtype=float32), array(0.8276, dtype=float32), array(0.8416, dtype=float32), array(0.849, dtype=float32), array(0.852, dtype=float32), array(0.8624, dtype=float32), array(0.8716, dtype=float32), array(0.8744, dtype=float32), array(0.874, dtype=float32), array(0.8812, dtype=float32), array(0.8828, dtype=float32), array(0.8786, dtype=float32), array(0.8838, dtype=float32)]


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
# Create the plot
plt.figure(figsize=(10, 6))

# Plot training and validation accuracies
plt.plot(train_acc, label='Training Accuracy', color='blue', marker='o')
plt.plot(val_acc, label='Validation Accuracy', color='orange', marker='x')

# Add labels and title
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.title('Training and Validation Accuracy over Epochs')

# Show legend
plt.legend()

# Display the plot
plt.grid(True)
plt.show()